In [1]:
import pandas as pd

In [2]:
data = r'C:\Users\YangTeng\Desktop\工作\品牌号整合\Joanne【模板2】.xlsx'# 输入带查重文件路径

brand_first_data = pd.read_excel(data,sheet_name='品牌1',dtype=str).fillna('')

brand_second_data = pd.read_excel(data,sheet_name='品牌2',dtype=str).fillna('')

brand_third_data = pd.read_excel(data,sheet_name='品牌3',dtype=str).fillna('')

#### -----------------------------------------------------
定义清理差分oe函数

In [3]:
def Separate_oe(brand_date):
    
    clean_oe_data = brand_date[brand_date['OE']!='']

    new_brand_oe_data = clean_oe_data[['brand','OE']].copy()
    new_brand_oe_data['OE'] = new_brand_oe_data['OE'].str.split(';')
    new_brand_oe_data = new_brand_oe_data.explode('OE')
    
    return new_brand_oe_data


### -------------------------------------------

In [4]:
new_brand_first_data = Separate_oe(brand_first_data)
new_brand_first_data

,brand,OE
0,38106,10069954
1,38103,10239670
1,38103,12456152
1,38103,12561094
1,38103,12580296
...,...,...
445,39467,JR2E6B209AA
445,39467,JR3E6B209AA
445,39467,JR3Z6B209A
446,39518,1662024020


In [5]:
new_brand_second_data = Separate_oe(brand_second_data)
new_brand_second_data

,brand,OE
0,419000,0166200H020
0,419000,166200H021
0,419000,166200H030
0,419000,1662028040
0,419000,1662028041
...,...,...
106,419404,6422001370
106,419404,68040206AA
107,419422,25191534
107,419422,55563512


In [6]:
new_brand_third_data = Separate_oe(brand_third_data)
new_brand_third_data

,brand,OE
0,89206,E7SE6B209AA
0,89206,E7SE6B209AB
0,89206,E7SE6B209DB
0,89206,E7SZ6B209B
0,89206,F1ZE6B209CA
...,...,...
260,89663,12581203
263,89667,89667FN
263,89667,MD367192
268,89665,AL3Z6B209A


### --------------------------------------------
定义连接函数

In [7]:
def merge_brand(brand_first,brand_second):
    
    merge_data = pd.merge(brand_first,brand_second,on='OE',how='left').fillna('')
    
    repeat_data = merge_data[merge_data['brand_y']!='']
    
    OE_repeat_data = repeat_data.groupby(['brand_x'],as_index=False,sort=False).agg({'OE':';'.join,'brand_y':';'.join})
    
    OE_repeat_data['OE'] = OE_repeat_data['OE'].str.split(';').apply(lambda x: sorted(list(set(x)), key=x.index))
    OE_repeat_data['brand_y'] = OE_repeat_data['brand_y'].str.split(';').apply(lambda x: sorted(list((set(x))), key=x.index))
    
    OE_repeat_data.rename(columns={'brand_x':'brand','OE':'品牌重复OE','brand_y':'重复品牌号'},inplace=True)
    
    OE_repeat_data['品牌重复OE'] = OE_repeat_data['品牌重复OE'].apply(lambda x: ';'.join(x))
    OE_repeat_data['重复品牌号'] = OE_repeat_data['重复品牌号'].apply(lambda x: ';'.join(x))
    
#     result = pd.merge(brand_first,OE_repeat_data,on='brand',how='left').fillna('')
    
    return OE_repeat_data

### ----------------------------------------

#### ------------------------------------------
品牌1与品牌2连接

In [8]:
OE_repeat_data_12 = merge_brand(new_brand_first_data,new_brand_second_data)
OE_repeat_data_12

result_12= pd.merge(brand_first_data,OE_repeat_data_12,on='brand',how='left').fillna('')
result_12

,brand,OE,品牌重复OE,重复品牌号
0,38106,10069954,,
1,38103,10239670;12456152;12561094;12580296;31025168;8...,89017309,419100
2,38104,12563083;24503851;24507667;24507684,12563083,419104
3,38105,10044970;10078911;10114908;10166387;14091898;1...,,
4,38102,F37Z6B209A;F3TZ6B209C;F47E6B209AA;F4TE6B209AA;...,F77Z6B209AA;ZZM515980;ZZM815980,419200
...,...,...,...,...
443,39428,68201164AA,,
444,39516,12647765,,
445,39467,2147525;CR3E6B209AB;JR2E6B209AA;JR3E6B209AA;JR...,,
446,39518,1662024020,,


#### ------------------------------------------
品牌1与品牌3连接

In [9]:
OE_repeat_data_13 = merge_brand(new_brand_first_data,new_brand_third_data)
OE_repeat_data_13

result_13 = pd.merge(brand_first_data,OE_repeat_data_13,on='brand',how='left').fillna('')
result_13

,brand,OE,品牌重复OE,重复品牌号
0,38106,10069954,10069954,89227
1,38103,10239670;12456152;12561094;12580296;31025168;8...,10239670;12456152;12561094;12580296;89017309,89231
2,38104,12563083;24503851;24507667;24507684,12563083;24503851;24507667;24507684,89241
3,38105,10044970;10078911;10114908;10166387;14091898;1...,10044970;10078911;10114908;10166387;14091898;1...,89203
4,38102,F37Z6B209A;F3TZ6B209C;F47E6B209AA;F4TE6B209AA;...,F37Z6B209A;F3TZ6B209C;F47E6B209AA;F4TE6B209AA;...,89230;89393
...,...,...,...,...
443,39428,68201164AA,,
444,39516,12647765,,
445,39467,2147525;CR3E6B209AB;JR2E6B209AA;JR3E6B209AA;JR...,,
446,39518,1662024020,,


In [25]:
result_1 = pd.merge(result_12,result_13,on='brand',how='left').fillna('')

result_1.rename(columns={'OE_x':'OE','品牌重复OE_x':'品牌2重复oe','重复品牌号_x':'重复品牌号2','品牌重复OE_y':'品牌3重复oe','重复品牌号_y':'重复品牌号3'},inplace=True)

result_1 = result_1.drop(['OE_y'],axis=1)

result_1

,brand,OE,品牌2重复oe,重复品牌号2,品牌3重复oe,重复品牌号3
0,38106,10069954,,,10069954,89227
1,38103,10239670;12456152;12561094;12580296;31025168;8...,89017309,419100,10239670;12456152;12561094;12580296;89017309,89231
2,38104,12563083;24503851;24507667;24507684,12563083,419104,12563083;24503851;24507667;24507684,89241
3,38105,10044970;10078911;10114908;10166387;14091898;1...,,,10044970;10078911;10114908;10166387;14091898;1...,89203
4,38102,F37Z6B209A;F3TZ6B209C;F47E6B209AA;F4TE6B209AA;...,F77Z6B209AA;ZZM515980;ZZM815980,419200,F37Z6B209A;F3TZ6B209C;F47E6B209AA;F4TE6B209AA;...,89230;89393
...,...,...,...,...,...,...
443,39428,68201164AA,,,,
444,39516,12647765,,,,
445,39467,2147525;CR3E6B209AB;JR2E6B209AA;JR3E6B209AA;JR...,,,,
446,39518,1662024020,,,,


#### -------------------------------------------------
与品牌1的连接结束

#### ------------------------------------------
定义去除重复品牌号函数

In [ ]:
def Repeat_oe(brand):
    brand_list = 

In [27]:
brand = result_1['重复品牌号2']!


for i in brand:
    brand_number['重复品牌号2'] = brand['重复品牌号2'].str.split(';')
    brand_number = brand_number.explode('重复品牌号2')

KeyError: '重复品牌号2'

In [32]:
brand = result_1[result_1['重复品牌号2'] !='']['重复品牌号2']
brand

1             419100
2             419104
4             419200
5             419102
6             419113
           ...      
356           419318
370           419358
374    419008;419010
398           419040
428           419112
Name: 重复品牌号2, Length: 105, dtype: object

In [ ]:
brand_list = []
for i in brand:
    brand_list = brand['重复品牌号2'].str.split(';')
    brand_number = brand_number.explode('重复品牌号2')